# Rule-Based LabelEngine Validation Workbench

This notebook validates the **LabelEngine** and **MarketStateLabeler** within the Forex_DNN quantitative trading framework. It demonstrates:
1. Deterministic, rule-based sliding window sample generation.
2. Comparison of different window sizes (e.g., 20, 25, 35, 50, 75).
3. Quality and consistency checks using `DatasetValidator`.
4. Clean visualizations of class distribution and label alignment with market structure using **Matplotlib**.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

# Ensure imports work
sys.path.append(os.path.abspath('.'))

from ML.label_engine import LabelEngine
from ML.market_state_labeler import MarketStateLabeler
from ML.dataset_validator import DatasetValidator
from ML.generate_dataset import generate_synthetic_candles

## 1. Load Data

We generate deterministic synthetic candle data to simulate active market conditions (trends, ranges, and transitions).

In [ ]:
df_raw = generate_synthetic_candles(num_bars=1000, seed=123)
print(f"Generated raw dataset size: {df_raw.shape}")
df_raw.head()

## 2. Compare Different Window Sizes

We evaluate the LabelEngine's performance and output distribution across multiple sliding window sizes (20, 25, 35, 50, and 75).

In [ ]:
window_sizes = [20, 25, 35, 50, 75]
datasets = {}
validation_reports = {}

for size in window_sizes:
    print(f"\n--- Generating Dataset for Window Size = {size} ---")
    labeler = MarketStateLabeler(atr_period=14)
    engine = LabelEngine(window_size=size, window_stride=1, labeler=labeler)
    
    df_labeled = engine.generate_dataset(
        data_inputs=df_raw,
        symbol="EURUSD",
        timeframe="M15"
    )
    
    # Validate
    validator = DatasetValidator()
    report = validator.validate(df_labeled, expected_window_size=size)
    
    datasets[size] = df_labeled
    validation_reports[size] = report
    
    print(f"Total Windows: {engine.total_windows_processed}")
    print(f"Labeled Samples: {len(df_labeled)}")
    print(f"Removed (ambiguous): {engine.removed_samples_count}")
    print(f"Validation Passed: {report['is_valid']}")
    print(f"Class Distribution: {report['metrics'].get('class_distribution')}")

## 3. Visualize Class Distribution Comparison

We plot how the class counts change as the window size increases. A larger window size generally leads to fewer TREND or RANGE matches due to more stringent criteria over longer time horizons.

In [ ]:
fig, axes = plt.subplots(1, len(window_sizes), figsize=(20, 5), sharey=True)
fig.suptitle("Class Distribution across Different Window Sizes", fontsize=16, y=1.05)

classes = ["TREND", "RANGE", "TRANSITION"]
colors = ["#2ca02c", "#d62728", "#1f77b4"]

for idx, size in enumerate(window_sizes):
    ax = axes[idx]
    report = validation_reports[size]
    counts = report["metrics"]["class_distribution"]
    
    vals = [counts.get(c, 0) for c in classes]
    
    bars = ax.bar(classes, vals, color=colors, edgecolor='black', alpha=0.85)
    ax.set_title(f"Window Size {size}")
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Label bars with values
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Visual Validation of Label Alignment

Let's visualize a slice of the dataset to overlay our generated target state labels with the underlying price action and EMAs to ensure intuitive correctness.

In [ ]:
# Load default size 35 dataset
df_res = datasets[35]

# Select a slice of 100 windows to plot
plot_slice = df_res.iloc[200:300].copy()
plot_slice['idx'] = np.arange(len(plot_slice))

plt.figure(figsize=(15, 8))
plt.plot(plot_slice['idx'], plot_slice['Close'], label='Close Price', color='black', alpha=0.6, linewidth=1.5)
if 'ema_50' in plot_slice.columns:
    plt.plot(plot_slice['idx'], plot_slice['ema_50'], label='EMA 50', color='blue', linestyle='--', alpha=0.8)
if 'ema_600' in plot_slice.columns:
    plt.plot(plot_slice['idx'], plot_slice['ema_600'], label='EMA 600', color='red', linestyle='--', alpha=0.8)

# Group and highlight regions
targets = plot_slice['target'].values
indices = plot_slice['idx'].values

for i in range(len(plot_slice)):
    lbl = targets[i]
    idx_val = indices[i]
    if lbl == 'TREND':
        plt.axvspan(idx_val - 0.5, idx_val + 0.5, color='green', alpha=0.15, label='TREND' if i == 0 or 'TREND' not in plt.gca().get_legend_handles_labels()[1] else "")
    elif lbl == 'RANGE':
        plt.axvspan(idx_val - 0.5, idx_val + 0.5, color='red', alpha=0.15, label='RANGE' if i == 0 or 'RANGE' not in plt.gca().get_legend_handles_labels()[1] else "")
    elif lbl == 'TRANSITION':
        plt.axvspan(idx_val - 0.5, idx_val + 0.5, color='blue', alpha=0.1, label='TRANSITION' if i == 0 or 'TRANSITION' not in plt.gca().get_legend_handles_labels()[1] else "")

plt.title("LabelEngine Rule-Based Regimes Overlaid on Price Action (Window Size 35)", fontsize=14)
plt.xlabel("Sample Index")
plt.ylabel("Price")
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

## Conclusion

The LabelEngine successfully generates high-quality, reproducible rule-based targets suitable for training the `MarketStateClassifier` without manual label noise, zero look-ahead bias, and with detailed traceability.